# 03 · Differential expression (R, DESeq2)
kallisto → gene-level counts (tximport) → DESeq2. **Aging and rejuvenation cohorts are modelled separately** (strong cohort/batch effect), and each rejuvenation treatment against its own control (controls differ in sex).

| Dataset | Samples | Model | Contrasts |
|---|---|---|---|
| Aging | 12 (3/18/21 mo, 2♂+2♀ each) | `~ sex + group` | 18 vs 3, 21 vs 3 |
| aDR | 9 ♀ (AL 4, aDR 5) | `~ group` | aDR vs AL |
| YMP | 6 ♂ (PBS 3, YMP 3) | `~ group` | YMP vs PBS |

Kernel: **R** (`ir`) from the pixi environment. Results go to `results/de/`.

In [ ]:
source('../R/helpers.R')
dir.create(file.path(ROOT, 'results/de'), showWarnings = FALSE)
samples <- load_samples()
x <- load_txi(rownames(samples))
txi <- x$txi; genes <- x$genes
cat(nrow(txi$counts), 'genes x', ncol(txi$counts), 'samples\n')
dplyr::count(samples, cohort, group, sex)

## 1 · All 27 samples: cohort effect
Expect PC1 to separate the two cohorts, which is why they are analysed apart.

In [ ]:
dds_all <- DESeqDataSetFromTximport(txi, samples, ~ 1)
dds_all <- dds_all[rowSums(counts(dds_all) >= 10) >= 4, ]
pca_plot(dds_all, 'cohort', 'sex')

## 2 · Aging cohort (3 vs 18 vs 21 months)
### 2a · Outlier check
One 21-month female (SRR21355389) dominates PC1 when all 12 aging samples are used.

In [ ]:
s_age_all <- samples %>% filter(cohort == 'aging') %>% mutate(group = droplevels(group))
dds_age_all <- make_dds(txi, s_age_all, ~ sex + group, min_n = 4)
pca_plot(dds_age_all, 'group', 'sex')

In [ ]:
outlier_genes(dds_age_all, 'SRR21355389', genes)

Its top genes are **hemoglobins** (Hbb, Hba), **neutrophil** genes (S100a8/9, Ngp, Camp), **collagens** (Col1a1/2) and Igkc: blood/meningeal contamination (incomplete perfusion), not aging biology.

**Decision:** exclude it (edit `EXCLUDE` to change); 2b shows what this changes.

In [ ]:
EXCLUDE <- 'SRR21355389'   # set to character(0) to keep all 12 aging samples
s_age <- s_age_all[!rownames(s_age_all) %in% EXCLUDE, ]
dds_age <- make_dds(txi, s_age, ~ sex + group, min_n = 3)
pca_plot(dds_age, 'group', 'sex')

### 2b · DE and sensitivity to the exclusion

In [ ]:
age <- list(`18v3` = de_table(dds_age, c('group', 'aging 18mo', 'aging 3mo'), genes),
            `21v3` = de_table(dds_age, c('group', 'aging 21mo', 'aging 3mo'), genes))
bind_rows(lapply(age, n_deg), .id = 'contrast')

In [ ]:
# a = outlier excluded (used), b = all 12 samples
age_all_21v3 <- de_table(dds_age_all, c('group', 'aging 21mo', 'aging 3mo'), genes)
compare_de(age[['21v3']], age_all_21v3)

In [ ]:
volcano(age[['21v3']], 'Aging: 21 vs 3 months (cerebellum)')

## 3 · Rejuvenation: aDR vs AL (females) and YMP vs PBS (males)

In [ ]:
s_adr <- samples %>% filter(group %in% c('AL', 'aDR')) %>% mutate(group = droplevels(group))
s_ymp <- samples %>% filter(group %in% c('PBS', 'YMP')) %>% mutate(group = droplevels(group))
dds_adr <- make_dds(txi, s_adr, ~ group, min_n = 4)
dds_ymp <- make_dds(txi, s_ymp, ~ group, min_n = 3)
rej <- list(aDR = de_table(dds_adr, c('group', 'aDR', 'AL'), genes),
            YMP = de_table(dds_ymp, c('group', 'YMP', 'PBS'), genes))
bind_rows(lapply(rej, n_deg), .id = 'contrast')

In [ ]:
pca_plot(dds_adr, 'group')

In [ ]:
pca_plot(dds_ymp, 'group')

In [ ]:
volcano(rej$aDR, 'aDR vs AL (cerebellum)')

In [ ]:
volcano(rej$YMP, 'YMP vs PBS (cerebellum)')

## 4 · Save
One CSV per contrast (all tested genes: DESeq2 columns + symbol) for notebook 04.

In [ ]:
res <- c(setNames(age, paste0('aging_', names(age))), rej)
invisible(imap(res, \(r, n) write_csv(r, file.path(ROOT, 'results/de', paste0(n, '.csv')))))
list.files(file.path(ROOT, 'results/de'))